In [1]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBRegressor

from sklearn.model_selection import (
    TimeSeriesSplit,
    RandomizedSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

import joblib
import warnings

warnings.filterwarnings("ignore")

In [2]:
X_train = joblib.load("Demand_X_train.pkl")
X_test = joblib.load("Demand_X_test.pkl")

y_train = joblib.load("Demand_y_train.pkl")
y_test = joblib.load("Demand_y_test.pkl")

In [3]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)

print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (100, 11)
X_test  : (26, 11)
y_train : (100,)
y_test  : (26,)


In [4]:
# ============================
# Improved Hyperparameter Tuning - XGBoost
# ============================

xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)

param_grid = {

    "n_estimators": [100, 200, 300, 500],

    "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],

    "max_depth": [2, 3, 4, 5],

    "min_child_weight": [1, 3, 5, 7],

    "subsample": [0.7, 0.8, 0.9, 1.0],

    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],

    "gamma": [0, 0.1, 0.3, 0.5, 1],

    "reg_alpha": [0, 0.01, 0.1, 1],

    "reg_lambda": [1, 2, 5, 10]
}

tscv = TimeSeriesSplit(n_splits=5)

random_search = RandomizedSearchCV(

    estimator=xgb,

    param_distributions=param_grid,

    n_iter=60,

    scoring="r2",

    cv=tscv,

    random_state=42,

    verbose=1,

    n_jobs=-1

)

random_search.fit(X_train, y_train)

print("Best Parameters:")
print(random_search.best_params_)

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best Parameters:
{'subsample': 0.7, 'reg_lambda': 10, 'reg_alpha': 1, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.08, 'gamma': 0.3, 'colsample_bytree': 1.0}


In [5]:
# ============================
# Train the Optimized XGBoost Model
# ============================

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,

    n_estimators=300,
    max_depth=4,
    learning_rate=0.1,

    subsample=0.8,
    colsample_bytree=1.0,

    min_child_weight=7,
    gamma=0,

    reg_alpha=1,
    reg_lambda=10
)

xgb_model.fit(X_train, y_train)

print("Optimized XGBoost model trained successfully!")

Optimized XGBoost model trained successfully!


In [6]:
# ============================
# Make Predictions
# ============================

# Training Predictions
y_train_pred = xgb_model.predict(X_train)

# Testing Predictions
y_test_pred = xgb_model.predict(X_test)

print("Predictions generated successfully!")

Predictions generated successfully!


In [7]:
# ============================
# Model Evaluation
# ============================

# Training Metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mape = mean_absolute_percentage_error(y_train, y_train_pred) * 100
train_r2 = r2_score(y_train, y_train_pred)

# Testing Metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mape = mean_absolute_percentage_error(y_test, y_test_pred) * 100
test_r2 = r2_score(y_test, y_test_pred)

print("=" * 50)
print("Training Performance")
print("=" * 50)
print(f"MAE  : {train_mae:.2f}")
print(f"RMSE : {train_rmse:.2f}")
print(f"MAPE : {train_mape:.2f}%")
print(f"R²   : {train_r2:.4f}")

print("\n")

print("=" * 50)
print("Testing Performance")
print("=" * 50)
print(f"MAE  : {test_mae:.2f}")
print(f"RMSE : {test_rmse:.2f}")
print(f"MAPE : {test_mape:.2f}%")
print(f"R²   : {test_r2:.4f}")

Training Performance
MAE  : 20.30
RMSE : 46.69
MAPE : 0.24%
R²   : 0.9975


Testing Performance
MAE  : 352.81
RMSE : 564.76
MAPE : 3.04%
R²   : 0.6701


In [8]:
# ============================
# XGBoost Feature Importance
# ============================

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgb_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

               Feature  Importance
2   Electricity_Supply    0.898489
3     Solar_Irradiance    0.019958
10        Demand_Lag_3    0.017066
5                 Year    0.013890
8         Demand_Lag_1    0.009957
1             Rainfall    0.009028
0             Humidity    0.008422
7            Month_cos    0.007072
4          Temperature    0.006652
9         Demand_Lag_2    0.005549
6            Month_sin    0.003917
